# Hierarchical review and rejection diagnostics

This notebook analyzes Olga's event reviews as a two-level problem: first **VALID vs REJECTED**, then **Logging vs Wildfire** inside accepted events. It reuses stored model labels and the latest saved latent-diagnostic table; RemoteCLIP embeddings and inference are not recomputed.

## Analysis contract

- Raw reviewer comments remain unchanged. Deterministic reason groups are derived only for analysis.
- Rejected events are not binary-classification errors. They form a third human state.
- Knownness and HDBSCAN analyses include only events present in the saved latent table. Missing latent coverage is reported explicitly.
- Events from the same point are dependent. The exploratory validity models use grouped cross-validation by `sample_id`.
- Thresholds are transferred from the MANUAL_TRAIN knownness reference and are not optimized on Olga's reviews.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display
from sqlalchemy import create_engine
from sqlalchemy.orm import Session

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "Scripts" / "ResearchTool").exists()
)
SCRIPTS_ROOT = PROJECT_ROOT / "Scripts"
if str(SCRIPTS_ROOT) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_ROOT))

from ResearchScripts.ml_tests.config import load_config
from ResearchScripts.ml_tests.rejection_diagnostics import (
    find_latest_latent_run,
    rejection_diagnostic_figures,
    run_rejection_diagnostics,
)
from ResearchScripts.ml_tests.year_mismatch_research import (
    year_mismatch_ranking_figure,
    year_mismatch_timeline_figure,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 140)

## Parameters

Set `LATENT_RUN_DIR` to a specific completed latent run for a frozen comparison. With `None`, the most recent run containing `latent_event_level_table.csv` is selected.

In [ ]:
REVIEWER_NAME = "Olga"
PROFILE = "quick"
RANDOM_SEED = 42
LATENT_RUN_DIR = None

config = load_config(profile=PROFILE)
database_path = config.path_value("database").resolve()
ml_output_root = config.path_value("output_root").resolve()
latent_run_dir = Path(LATENT_RUN_DIR).resolve() if LATENT_RUN_DIR else find_latest_latent_run(ml_output_root)
report_output_root = ml_output_root / "rejection_diagnostics"
database_path, latent_run_dir, report_output_root

## Run the joined review audit

The database is opened read-only from the notebook's perspective. A timestamped artifact directory is created for tables, plots, and `REPORT.md`.

In [ ]:
engine = create_engine(f"sqlite:///{database_path.as_posix()}", future=True)
with Session(engine) as db:
    result = run_rejection_diagnostics(
        db,
        reviewer_name=REVIEWER_NAME,
        latent_run_dir=latent_run_dir,
        output_root=report_output_root,
        seed=RANDOM_SEED,
    )

print(f"Report directory: {result['run_dir']}")
display(pd.DataFrame([result["summary"]]).T.rename(columns={0: "value"}))

## 1. Normalized review hierarchy

The analytical hierarchy is `VALID_LOGGING`, `VALID_WILDFIRE`, `NO_PRE`, `QC_FAILED`, `YEAR_MISMATCH`, `EVENT_TOO_SMALL`, `NO_VISIBLE_CHANGE`, `TYPE_UNCLEAR`, and `OTHER_INVALID`. The mapping table preserves every original rejection comment.

In [ ]:
display(pd.DataFrame(result["hierarchy_counts"]))
display(pd.DataFrame(result["raw_reason_mapping"]))

## 2. Logging / Wildfire / Rejected closed-set audit

The classifiers have no Rejected output. These tables therefore show where each binary closed-set model sends events that a human considers unsuitable for classification.

In [ ]:
three_state = pd.DataFrame(result["three_state_tables"])
display(three_state)
for model_name in three_state["model"].unique():
    display(Markdown(f"### {model_name}"))
    display(three_state[three_state["model"] == model_name].pivot(index="predicted_label", columns="human_state", values="count"))

## 3. Model probabilities on rejected events

Probability and normalized binary entropy are compared across human Logging, human Wildfire, and Rejected. A high extreme-probability share for Rejected means binary confidence does not act as an invalid/OOD signal.

In [ ]:
display(pd.DataFrame(result["probability_summaries"]))

## 4. Knownness and rejection reasons

Knownness is compared first by review status and then by normalized rejection reason. This tests whether human invalidity is equivalent to latent-space out-of-distribution behavior.

In [ ]:
display(pd.DataFrame(result["knownness_status_summary"]))
display(pd.DataFrame(result["rejection_reason_summary"]))

## 5. Does HDBSCAN noise identify human rejection?

The table reports conditional noise rates, an odds ratio, and a Fisher exact test. For Hansen-mined rows the current pipeline inherits HDBSCAN status from the nearest manual event, so this remains a descriptive diagnostic.

In [ ]:
display(pd.DataFrame([result["hdbscan_rejection"]]).T.rename(columns={0: "value"}))

## 6. RemoteCLIP error audit

Accepted RemoteCLIP errors are listed with knownness, cluster role, class similarities, P(Wildfire), event area, image counts, and reviewer comments.

In [ ]:
display(pd.DataFrame(result["remoteclip_errors"]))

## 7. Model disagreement as review priority

Each prediction pair is described by its VALID, REJECTED, human Logging, and human Wildfire shares.

In [ ]:
display(pd.DataFrame(result["model_disagreement"]))

## 8. Point-level event history

The 102 event rows belong to only 49 points. Point-level aggregation identifies Logging-only, Wildfire-only, mixed, and rejected-only histories and tests whether each model reconstructs all accepted years correctly.

In [ ]:
display(pd.DataFrame(result["point_summary"]))
display(pd.DataFrame(result["point_rows"]))

## 9. Knownness reject coverage

For each transferred threshold the notebook reports reject precision, reject recall, retained coverage, lost human-valid events, and RemoteCLIP quality on retained valid events.

In [ ]:
display(pd.DataFrame(result["coverage_analysis"]))

## 10. Exploratory VALID vs REJECTED pre-filter

Logistic Regression and XGBoost use knownness, binary-model probabilities and margins, HDBSCAN noise, event area, PRE/POST counts, Hansen history, and multi-event status. This is a small grouped-CV diagnostic, not a production estimate.

In [ ]:
validity = result["validity_models"]
display(pd.DataFrame(validity.get("metrics", [])))
display(pd.DataFrame(validity.get("feature_importance", [])))

## 11. Per-reason feature distributions

The table keeps VALID as the reference and profiles event area, Hansen-loss share and pixels, PRE/EVENT/POST coverage, QC flags, knownness, both model probabilities, RemoteCLIP similarities, season, and event year. The focused views below make `EVENT_TOO_SMALL vs VALID` and `YEAR_MISMATCH vs VALID` explicit.

In [ ]:
display(pd.DataFrame(result["reason_feature_summary"]))
reason_distributions = pd.DataFrame(result["reason_feature_distributions"])
display(reason_distributions[reason_distributions["reason"].isin(["VALID", "EVENT_TOO_SMALL", "YEAR_MISMATCH"])])
display(pd.DataFrame(result["reason_qc_flag_counts"]))
display(pd.DataFrame(result["reason_season_counts"]))
display(pd.DataFrame(result["reason_year_counts"]))

## 12. EVENT_TOO_SMALL deterministic-threshold search

For event area, Hansen pixels, and Hansen-loss share inside the AOI, the sweep evaluates rules of the form `value <= threshold -> reject`. A candidate threshold must cover at least five events and achieve at least 70% reject precision; otherwise the notebook labels the best F1 threshold as exploratory only.

In [ ]:
small_event = result["small_event_analysis"]
display(pd.DataFrame(small_event["summary"]))
display(pd.DataFrame(small_event["bins"]))
display(pd.DataFrame(small_event["thresholds"]))

## 13. YEAR_MISMATCH temporal consistency

The analysis compares the Hansen year with an expert-visible year only when the free-text reason explicitly identifies a previous, next, or absolute year. It also checks Y-1/Y/Y+1 Hansen events and PRE/EVENT/POST scene years. The current schema has no structured expert year and no FIRMS table, so both gaps are reported rather than imputed.

In [ ]:
temporal = result["year_mismatch_analysis"]
display(pd.DataFrame([temporal["summary"]]).T.rename(columns={0: "value"}))
display(pd.DataFrame(temporal["events"]))

### 13.1. Can we select the main physical event year?

Each YEAR_MISMATCH point is expanded into a `Y-2 / Y-1 / Y / Y+1` timeline. Every Hansen event carries area, Hansen pixels and AOI share, RemoteCLIP/XGBoost `P(Wildfire)`, expert status, PRE/EVENT/POST coverage, QC information, season counts, and reserved FIRMS fields.

The target year is derived conservatively: an explicit expert-visible year in the rejection comment has priority; otherwise a single `VALID_WILDFIRE` year is used. Points with several equally valid years or no selectable year remain outside evaluation. Deterministic rankings are compared with leave-one-point-out Logistic Regression, so no event from the tested point enters its training fold.

In [ ]:
year_branch = result["year_mismatch_research"]
display(pd.DataFrame([year_branch["summary"]]).T.rename(columns={0: "value"}))
display(pd.DataFrame(year_branch["point_summaries"]))
display(pd.DataFrame(year_branch["timeline_wide"]))
display(pd.DataFrame(year_branch["ranking_metrics"]))
display(pd.DataFrame(year_branch["point_predictions"]))

In [ ]:
timeline_figure = year_mismatch_timeline_figure(year_branch)
ranking_figure = year_mismatch_ranking_figure(year_branch)
if timeline_figure is not None:
    display(timeline_figure)
if ranking_figure is not None:
    display(ranking_figure)

## 14. NO_PRE and QC_FAILED as acquisition-pipeline statistics

These reasons are not treated as an ML classification task. The tables quantify losses caused by missing temporal coverage and QC, split by event year, acquisition season, and QC flag.

In [ ]:
pipeline_stats = result["acquisition_pipeline"]
display(pd.DataFrame(pipeline_stats["summary"]))
display(pd.DataFrame(pipeline_stats["by_year"]))
display(pd.DataFrame(pipeline_stats["by_season"]))
display(pd.DataFrame(pipeline_stats["qc_flags"]))

## 15. Rejection reason vs model agreement

The conditional tables answer questions such as `P(EVENT_TOO_SMALL | RC != XGB)` and `P(YEAR_MISMATCH | RC = XGB = Wildfire)` without conflating missing model labels with agreement.

In [ ]:
reason_agreement = result["reason_model_agreement"]
display(pd.DataFrame([reason_agreement["headline"]]).T.rename(columns={0: "value"}))
display(pd.DataFrame(reason_agreement["cross_table"]))
display(pd.DataFrame(reason_agreement["conditionals"]))

## 16. One-vs-VALID exploratory classifiers

Grouped logistic regressions test whether `EVENT_TOO_SMALL`, `YEAR_MISMATCH`, and `NO_VISIBLE_CHANGE` are distinguishable from VALID using currently available features. Reasons with fewer than five examples are reported as insufficient instead of producing unstable metrics.

In [ ]:
one_vs_valid = result["one_vs_valid"]
display(pd.DataFrame(one_vs_valid["metrics"]))
display(pd.DataFrame(one_vs_valid["coefficients"]))

## 17. Point-level reason co-occurrence and temporal transitions

Events are grouped by point to show which review outcomes coexist and which outcomes follow one another across adjacent Hansen years. This exposes cases where one year is rejected as `YEAR_MISMATCH` while a neighboring year at the same point is valid.

In [ ]:
point_relationships = result["point_reason_relationships"]
display(pd.DataFrame(point_relationships["cooccurrence"]))
display(pd.DataFrame(point_relationships["adjacent_transitions"]))
display(pd.DataFrame(point_relationships["histories"]))

## Figures

The same figures are saved under the run's `plots` directory.

In [ ]:
figures = rejection_diagnostic_figures(result)
for figure in figures.values():
    display(figure)

## Generated report

Rerunning the notebook creates a new timestamped directory and leaves previous runs unchanged.

In [ ]:
report_text = Path(result["report_path"]).read_text(encoding="utf-8")
display(Markdown(report_text))